In [1]:
import pickle
import numpy as np
import plotly.express as px

In [2]:
import cv2
import numpy as np
import random
import warnings
import argparse
from pathlib import Path


In [3]:
import torch

In [ ]:
questions = torch.tensor([[0,0,1], [0,1,0], [0,0,1]])




In [33]:
questions[:,2] == 1

tensor([ True, False,  True])

In [4]:
a = torch.arange(16).view((2,4,2))
print(a)
print(a.permute(0, 2, 1))

tensor([[[ 0,  1],
         [ 2,  3],
         [ 4,  5],
         [ 6,  7]],

        [[ 8,  9],
         [10, 11],
         [12, 13],
         [14, 15]]])
tensor([[[ 0,  2,  4,  6],
         [ 1,  3,  5,  7]],

        [[ 8, 10, 12, 14],
         [ 9, 11, 13, 15]]])


In [5]:
dict_1 = {'hi': [1,2,3], 'bye': [1,2]}

next(iter(dict_1.values()))

[1, 2, 3]

In [6]:
# constants:

QUESTION_SIZE = 18 # 2 x (6 for one-hot vector of color), 3 for question type, 3 for question subtype
Q_TYPE_IDX = 12
SUB_Q_TYPE_IDX = 15
ANSWER_SIZE = 10 # Answer : [yes, no, rectangle, circle, r, g, b, o, k, y]

COLORS = [
    (0,0,255),##r
    (0,255,0),##g
    (255,0,0),##b
    (0,156,255),##o
    (128,128,128),##k
    (0,255,255)##y
]

In [7]:
def center_generate(
        objects: list[np.ndarray], obj_size: int, img_size: int
        ) -> np.ndarray:
    
    while True:
        pas = True
        center = np.random.randint(0+obj_size, img_size - obj_size, 2)        
        if len(objects) > 0:
            for _, c, _ in objects:
                if ((center - c) ** 2).sum() < ((obj_size * 2) ** 2):
                    pas = False
        if pas:
            return center

In [8]:
def generate_ternary_question(
        objects: list[np.ndarray], t_subtype: int
        ) -> tuple[np.ndarray, int]:

    question = np.zeros((QUESTION_SIZE))
    rnd_colors = np.random.permutation(np.arange(5))
    # 1st object
    color1 = rnd_colors[0]
    question[color1] = 1
    # 2nd object
    color2 = rnd_colors[1]
    question[6 + color2] = 1

    question[Q_TYPE_IDX + 2] = 1
    
    if t_subtype >= 0 and t_subtype < 3:
        subtype = t_subtype
    else:
        subtype = random.randint(0, 2)

    question[subtype+SUB_Q_TYPE_IDX] = 1

    # get coordiantes of object from question
    A = objects[color1][1]
    B = objects[color2][1]

    if subtype == 0:
        """between->1~4"""

        between_count = 0 
        # check is any objects lies inside the box
        for other_obj in objects:
            # skip object A and B
            if (other_obj[0] == color1) or (other_obj[0] == color2):
                continue

            # Get x and y coordinate of third object
            other_objx = other_obj[1][0]
            other_objy = other_obj[1][1]

            if (A[0] <= other_objx <= B[0] and A[1] <= other_objy <= B[1]) or \
                (A[0] <= other_objx <= B[0] and B[1] <= other_objy <= A[1]) or \
                (B[0] <= other_objx <= A[0] and B[1] <= other_objy <= A[1]) or \
                (B[0] <= other_objx <= A[0] and A[1] <= other_objy <= B[1]):
                between_count += 1

        answer = between_count + 4
    elif subtype == 1:
        """is-on-band->yes/no"""
        
        grace_threshold = 12  # half of the size of objects
        epsilon = 1e-10  
        m = (B[1]-A[1])/((B[0]-A[0]) + epsilon ) # add epsilon to prevent dividing by zero
        c = A[1] - (m*A[0])

        answer = 1  # default answer is 'no'

        # check if any object lies on/close the line between object A and object B
        for other_obj in objects:
            # skip object A and B
            if (other_obj[0] == color1) or (other_obj[0] == color2):
                continue

            other_obj_pos = other_obj[1]
            
            # y = mx + c
            y = (m*other_obj_pos[0]) + c
            if (y - grace_threshold)  <= other_obj_pos[1] <= (y + grace_threshold):
                answer = 0
    elif subtype == 2:
        """count-obtuse-triangles->1~6"""

        obtuse_count = 0

        # disable warnings
        # the angle computation may fail if the points are on a line
        warnings.filterwarnings("ignore")
        for other_obj in objects:
            # skip object A and B
            if (other_obj[0] == color1) or (other_obj[0] == color2):
                continue

            # get position of 3rd object
            C = other_obj[1]
            # edge length
            a = np.linalg.norm(B - C)
            b = np.linalg.norm(C - A)
            c = np.linalg.norm(A - B)
            # angles by law of cosine
            alpha = np.rad2deg(np.arccos((b ** 2 + c ** 2 - a ** 2) / (2 * b * c)))
            beta = np.rad2deg(np.arccos((a ** 2 + c ** 2 - b ** 2) / (2 * a * c)))
            gamma = np.rad2deg(np.arccos((a ** 2 + b ** 2 - c ** 2) / (2 * a * b)))
            max_angle = max(alpha, beta, gamma)
            if max_angle >= 90 and max_angle < 180:
                obtuse_count += 1

        warnings.filterwarnings("default")
        answer = obtuse_count + 4

    return question, answer

In [9]:
def generate_binary_question(objects: list[np.ndarray]) -> tuple[np.ndarray, int]:
    
    question = np.zeros((QUESTION_SIZE))
    color = random.randint(0,5)
    question[color] = 1
    question[Q_TYPE_IDX+1] = 1
    subtype = random.randint(0,2)
    question[subtype+SUB_Q_TYPE_IDX] = 1

    if subtype == 0:
        """closest-to->rectangle/circle"""
        my_obj = objects[color][1]
        dist_list = [((my_obj - obj[1]) ** 2).sum() for obj in objects]
        dist_list[dist_list.index(0)] = 999
        closest = dist_list.index(min(dist_list))
        if objects[closest][2] == 'r':
            answer = 2
        else:
            answer = 3
            
    elif subtype == 1:
        """furthest-from->rectangle/circle"""
        my_obj = objects[color][1]
        dist_list = [((my_obj - obj[1]) ** 2).sum() for obj in objects]
        furthest = dist_list.index(max(dist_list))
        if objects[furthest][2] == 'r':
            answer = 2
        else:
            answer = 3

    elif subtype == 2:
        """count->1~6"""
        my_obj = objects[color][2]
        count = -1
        for obj in objects:
            if obj[2] == my_obj:
                count +=1 
        answer = count+4

    return question, answer

In [10]:
def generate_non_relational_question(
        objects: list[np.ndarray], img_size: int
        ) -> tuple[np.ndarray, int]:
    
    question = np.zeros((QUESTION_SIZE))
    color = random.randint(0,5)
    question[color] = 1
    question[Q_TYPE_IDX] = 1
    subtype = random.randint(0,2)
    question[subtype+SUB_Q_TYPE_IDX] = 1
    """Answer : [yes, no, rectangle, circle, r, g, b, o, k, y]"""
    if subtype == 0:
        """query shape->rectangle/circle"""
        if objects[color][2] == 'r':
            answer = 2
        else:
            answer = 3

    elif subtype == 1:
        """query horizontal position->yes/no"""
        if objects[color][1][0] < img_size / 2:
            answer = 0
        else:
            answer = 1

    elif subtype == 2:
        """query vertical position->yes/no"""
        if objects[color][1][1] < img_size / 2:
            answer = 0
        else:
            answer = 1

    return question, answer

In [11]:
def generate_sample(
        img_size: int, obj_size: int, nb_questions: int, t_subtype: int
        ):
    
    objects = []
    img = np.ones((img_size,img_size,3)) * 255

    # generate objects
    for color_id,color in enumerate(COLORS):  
        center = center_generate(objects, obj_size, img_size)
        if random.random()<0.5:
            start = (center[0]-obj_size, center[1]-obj_size)
            end = (center[0]+obj_size, center[1]+obj_size)
            cv2.rectangle(img, start, end, color, -1)
            objects.append((color_id,center,'r'))
        else:
            center_ = (center[0], center[1])
            cv2.circle(img, center_, obj_size, color, -1)
            objects.append((color_id,center,'c'))

    ternary_questions = []
    binary_questions = []
    nonrel_questions = []

    ternary_answers = []
    binary_answers = []
    nonrel_answers = []
    
    # generate questions
    for _ in range(nb_questions):
        ternary_q, ternary_a = generate_non_relational_question(objects, img_size)
        ternary_questions.append(ternary_q)
        ternary_answers.append(ternary_a)
        
        binary_q, binary_a = generate_binary_question(objects)
        binary_questions.append(binary_q)
        binary_answers.append(binary_a)

        nonrel_q, nonrel_a = generate_ternary_question(objects, t_subtype)
        nonrel_questions.append(nonrel_q)
        nonrel_answers.append(nonrel_a)
    
    img = img/255.
    return (
        img, 
        (ternary_questions, ternary_answers), 
        (binary_questions, binary_answers), 
        (nonrel_questions, nonrel_answers)
        )

In [12]:
def build_dataset(
        dataset_size: int, img_size: int, obj_size: int, 
        nb_questions: int, t_subtype: int
        ) -> dict[str, np.ndarray]:
    
    imgs = []
    ternary_questions, ternary_answers = [], []
    binary_questions, binary_answers = [], []
    nonrel_questions, nonrel_answers = [], []
    
    total_nb_questions = nb_questions*3
    image_idxs = np.arange(dataset_size).repeat(total_nb_questions)

    for _ in range(dataset_size):
        img, ternary, binary, nonrel = generate_sample(
            img_size, obj_size, nb_questions, t_subtype
            )
        imgs.append(img)
        ternary_questions.append(ternary[0])
        ternary_answers.append(ternary[1])
        binary_questions.append(binary[0])
        binary_answers.append(binary[1])
        nonrel_questions.append(nonrel[0])
        nonrel_answers.append(nonrel[1])

    
    return {
        'images': np.array(imgs),
        'image_idxs': image_idxs,
        'ternary_questions': np.array(ternary_questions),
        'ternary_answers': np.array(ternary_answers),
        'binary_questions': np.array(binary_questions), 
        'binary_answers': np.array(binary_answers), 
        'nonrel_questions': np.array(nonrel_questions),
        'nonrel_answers': np.array(nonrel_answers),
    }

def save_dataset(dataset: dict[str, np.ndarray], data_dir: Path, name: str):

    data_dir.mkdir(exist_ok=True, parents=True)
    file = data_dir / name
    np.savez_compressed(file, allow_pickle=True, **dataset)

    print(f'saved {name} to {str(str(data_dir.absolute()))}')

In [13]:
parser = argparse.ArgumentParser(description='Sort-of-CLEVR dataset generator')
parser.add_argument(
    '--seed', type=int, default=1, metavar='S', help='random seed (default: 1)'
    )
parser.add_argument(
    '--root', type=str, default='./data', help='dataset root directory'
    )
parser.add_argument(
    '--dir', type=str, default='sort-of-clevr', help='dataset directory'
)
parser.add_argument(
    '--train-size', type=int, default=9800, help='number of scenes generated for train dataset'
)
parser.add_argument(
    '--test-size', type=int, default=200, help='number of scenes generated for test dataset'
)
parser.add_argument(
    '--img-size', type=int, default=75, help='width and height for generated scenes'
)
parser.add_argument(
    '--obj-size', type=int, default=5, help='width and height for generated objects'
)
parser.add_argument(
    '--nb-questions', type=int, default=10, help='number of questions for each question type'
)
parser.add_argument(
    '--t-subtype', type=int, default=-1, help='Force ternary questions to be of a given type'
    )


_StoreAction(option_strings=['--t-subtype'], dest='t_subtype', nargs=None, const=None, default=-1, type=<class 'int'>, choices=None, required=False, help='Force ternary questions to be of a given type', metavar=None)

In [14]:

from dataclasses import dataclass
from pathlib import Path


@dataclass
class SortOfCLEVRConfig:
    seed: int = 1
    root: Path = Path('./data')
    dir: str = 'sort-of-clevr'
    train_size: int = 9800
    test_size: int = 200
    img_size: int = 75
    obj_size: int = 5
    nb_questions: int = 10
    t_subtype: int = -1

args = SortOfCLEVRConfig()

random.seed(args.seed)
np.random.seed(args.seed)

data_dir = Path(args.root) / args.dir
data_dir.mkdir(exist_ok=True, parents=True)

print('building test datasets...')
test_dataset = build_dataset(
    args.test_size, args.img_size, args.obj_size, 
    args.nb_questions, args.t_subtype
)
save_dataset(test_dataset, data_dir, f'test.npz')

print('building validation datasets...')
val_dataset = build_dataset(
    args.test_size, args.img_size, args.obj_size, 
    args.nb_questions, args.t_subtype
)
save_dataset(val_dataset, data_dir, f'val.npz')

print('building train datasets...')
train_dataset = build_dataset(
    args.train_size, args.img_size, args.obj_size, 
    args.nb_questions, args.t_subtype
)
save_dataset(train_dataset, data_dir, f'train.npz')

building test datasets...
saved test.npz to /home/nik/ImperialWork/msc_project/SyncNetProject/data/sort-of-clevr
building validation datasets...
saved val.npz to /home/nik/ImperialWork/msc_project/SyncNetProject/data/sort-of-clevr
building train datasets...
saved train.npz to /home/nik/ImperialWork/msc_project/SyncNetProject/data/sort-of-clevr


In [15]:
def get_sort_of_clevr_datasets(dir: str):
    train_path = np.load()

In [16]:
train_dataset = np.load('./data/sort-of-clevr/train.npz')


In [17]:
train_dataset['nonrel_questions'].shape

(9800, 10, 18)

In [18]:
train_dataset['images'][0].min()

np.float64(0.0)

In [19]:
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader
import torch
from typing import Iterable, Iterator

In [20]:
def load_sort_of_clevr(path: str):
    with np.load(path) as data:
            
        images = torch.from_numpy(data['images']).float() # (n_scenes, C, H, W)

        ternary_questions = torch.from_numpy(data['ternary_questions']).long() # (n_scenes, nb_questions, q_dim)
        ternary_answers = torch.from_numpy(data['ternary_answers']).long() # (n_scenes, nb_questions, a_dim)
        binary_questions = torch.from_numpy(data['binary_questions']).long() # (n_scenes, nb_questions, q_dim)
        binary_answers = torch.from_numpy(data['binary_answers']).long() # (n_scenes, nb_questions, a_dim)
        nonrel_questions = torch.from_numpy(data['nonrel_questions']).long() # (n_scenes, nb_questions, q_dim)
        nonrel_answers = torch.from_numpy(data['nonrel_answers']).long() # (n_scenes, nb_questions, a_dim)

        n_scenes = images.size(0)
        nb_questions = ternary_questions.size(1)
        total_nb_questions = 3*nb_questions
        q_dim = ternary_questions.size(-1)

        questions = torch.concatenate(
            (ternary_questions, binary_questions, nonrel_questions), dim=-2
            ).view(-1, q_dim) # (n_scenes, 3*nb_questions, q_dim) -> (n_scenes*3*nb_questions, q_dim)
        
        answers = torch.concatenate(
            (ternary_answers, binary_answers, nonrel_answers), dim=-2
            ).flatten() # (n_scenes, 3*nb_questions) -> (n_scenes*3*nb_questions)
        
    return (
        {'images': images, 'questions' :questions, 'answers': answers}, 
        n_scenes, total_nb_questions
        )

In [21]:
mode = 'train'

mode.capitalize()

'Train'

In [22]:
class SortOfClevrDataset(Dataset):

    def __init__(self, path: str) -> None:
        super().__init__()

        dataset, n_scenes, total_nb_questions = load_sort_of_clevr(path)
        
        self.images = dataset['images']
        self.questions = dataset['questions']
        self.answers = dataset['answers']
        self.n_scenes = n_scenes
        self.total_nb_questions = total_nb_questions
    
    def __len__(self):
        return self.n_scenes * self.total_nb_questions
    
    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        
        image_idx = idx // self.total_nb_questions
        return {
            'images': self.images[image_idx],
            'questions': self.questions[idx],
            'answers': self.answers[idx],

        }


In [23]:

def move_tensor_dict(batch: dict[str, torch.Tensor], device: str):
    non_blocking = (device == 'cuda')
    
    return {
        key: tensor.to(device, non_blocking=non_blocking) for key, tensor in batch.items()
        }

def batch_iter(
        dataloader: Iterator[dict[str, torch.Tensor]], device: str
        ) -> Iterable[dict[str, torch.Tensor]]:
    
    while True:
        
        for batch in dataloader:
            yield move_tensor_dict(batch, device)
        

In [24]:
def batch_iter_sort_of_clevr(
        path: str,
        batch_size: int,
        device: str,
        shuffle: bool=False,
        ) -> Iterator[dict[str, torch.Tensor]]:
    
    dataset, n_scenes, total_nb_questions = load_sort_of_clevr(path)
    dataset = move_tensor_dict(dataset, device)

    N = n_scenes*total_nb_questions

    while True:
        perm = torch.randperm(N, device=device)
        
        for i in range(0, N, batch_size):
            if shuffle:
                idx = perm[i: i+batch_size]
            else:
                end = min(N, i+batch_size)
                idx = torch.arange(i, end, device=device)

            image_idx = idx // total_nb_questions
            yield {
                'images': dataset['images'][image_idx], 
                'questions': dataset['questions'][idx], 
                'answers': dataset['answers'][idx]
                }

In [25]:
data = load_sort_of_clevr('./data/sort-of-clevr/train.npz')

In [26]:
data[0]['images'].shape

torch.Size([9800, 75, 75, 3])

In [27]:
data[0]['questions'].shape


torch.Size([294000, 18])

In [28]:
data[0]['answers'].shape

torch.Size([294000])

In [29]:
9800 * 30

294000

In [30]:
import time

device = 'cuda'
print(f'device: {device}')
start = time.perf_counter()

train_dataloader = step_iter_sort_of_clevr(
    './data/sort-of-clevr/train.npz', 
    256,
    device,
    True
)
steps = 10000
step = 0

while step < steps:
    batch = train_dataloader.__next__()
    step+=1
        

tot_time = time.perf_counter() - start

print(f'steps: {steps}')
print(f'steps per second: {(steps)/tot_time:.3f}')

device: cuda


NameError: name 'step_iter_sort_of_clevr' is not defined

In [ ]:
import time

print(f'device: {device}')
start = time.perf_counter()

train_dataset = SortOfClevrDataset('./data/sort-of-clevr/train.npz', device)

train_dataloader = DataLoader(
    train_dataset,
    256, 
    shuffle=True, 
    num_workers=0, 
)
steps = 10000
step = 0

data_iter = batch_iter(train_dataloader, device) # type: ignore
while step < steps:
    batch = iter(data_iter).__next__()
    step+=1

tot_time = time.perf_counter() - start

print(f'steps: {steps}')
print(f'steps per second: {(steps)/tot_time:.3f}')

device: cuda
steps: 10000
steps per second: 1332.798
